In [ ]:
!git clone https://github.com/ibrahimjohar/urdu-vlm-hallucination.git
import os
os.chdir('urdu-vlm-hallucination')

!pip install -q transformers accelerate bitsandbytes peft trl Pillow pandas requests

import torch
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
os.environ['HF_HOME'] = '/root/.cache/huggingface'

from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

#4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

#load model
print("loading LLaVA in 4-bit...")
model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")

#prepare and apply LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"VRAM used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from pathlib import Path
from PIL import Image
import requests

#dataset
class UrduVQADataset(Dataset):
    def __init__(self, json_path, processor, max_length=1024):
        with open(json_path, encoding='utf-8') as f:
            self.data = json.load(f)
        self.processor  = processor
        self.max_length = max_length
        self.coco_url   = "http://images.cocodataset.org/val2014/{}"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        conv     = sample['conversations_ur']
        question = conv[0]['value'].replace('<image>\n', '')
        answer   = conv[1]['value']

        image = Image.open(
            requests.get(self.coco_url.format(sample['image']),
                        stream=True, timeout=10).raw
        ).convert('RGB')

        conversation = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }, {
            "role": "assistant",
            "content": [{"type": "text", "text": answer}]
        }]

        prompt = self.processor.apply_chat_template(
            conversation, add_generation_prompt=False
        )
        inputs = self.processor(
            images=image,
            text=prompt,
            return_tensors='pt',
            max_length=self.max_length,
            truncation=True,
            padding='max_length'
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['labels'] = inputs['input_ids'].clone()
        return inputs

#config
FT_PATH = Path('data/processed/urdu_visual_qa_ft.json')
CHECKPOINT_DIR = Path('outputs/checkpoints/llava_qlora')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS     = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR         = 2e-4
SAVE_STEPS = 100

#load dataset
print("loading dataset...")
dataset = UrduVQADataset(FT_PATH, processor)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"samples         : {len(dataset)}")
print(f"steps per epoch : {len(loader)}")
print(f"total steps     : {len(loader) * EPOCHS}")

#optimizer
optimizer   = AdamW(model.parameters(), lr=LR)
model.train()
global_step = 0

print("\nstarting QLoRA training...")
print(f"epochs={EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}\n")

#training loop
for epoch in range(EPOCHS):
    epoch_loss  = 0
    num_batches = 0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        pixel_values = batch['pixel_values'].to('cuda', torch.float16)
        labels = batch['labels'].to('cuda')

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss  += outputs.loss.item()
        num_batches += 1
        global_step += 1

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()

        if global_step % 50 == 0:
            avg_loss = epoch_loss / num_batches
            print(f"epoch {epoch+1} | step {global_step} | loss {avg_loss:.4f}")

        if global_step % SAVE_STEPS == 0:
            ckpt_path = CHECKPOINT_DIR / f'checkpoint_step_{global_step}'
            model.save_pretrained(ckpt_path)
            print(f"checkpoint saved: {ckpt_path}")

    avg_epoch_loss = epoch_loss / num_batches
    print(f"\nepoch {epoch+1} complete | avg loss: {avg_epoch_loss:.4f}")

    ckpt_path = CHECKPOINT_DIR / f'checkpoint_epoch_{epoch+1}'
    model.save_pretrained(ckpt_path)
    print(f"epoch checkpoint saved: {ckpt_path}\n")

#save final
final_path = CHECKPOINT_DIR / 'final'
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"training complete. final model saved to: {final_path}")

In [ ]:
import os
import subprocess
result = subprocess.run(['find', '/kaggle/working', '-name', '*.safetensors', '-o', '-name', 'adapter_config.json'], 
                      capture_output=True, text=True)
print(result.stdout)

In [ ]:
import os
print(os.listdir('/kaggle/working'))
print(os.path.exists('/kaggle/working/outputs'))

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import login
login(token="...")

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
from peft import PeftModel
import torch

base = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(
    base, 'outputs/checkpoints/llava_qlora/final'
)
processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")

model.push_to_hub("ibrahimjohar/llava-1.5-7b-urdu-qlora")
processor.push_to_hub("ibrahimjohar/llava-1.5-7b-urdu-qlora")

In [5]:
#verify repo is cloned and data exists
import os
os.chdir('/kaggle/working/urdu-vlm-hallucination')
print(os.path.exists('data/processed/urdu_visual_qa_ft.json'))  # must print True

True


In [3]:
import os
os.chdir('/kaggle/working')
!git clone https://github.com/ibrahimjohar/urdu-vlm-hallucination.git
os.chdir('urdu-vlm-hallucination')
print(os.path.exists('data/processed/urdu_visual_qa_ft.json'))

fatal: destination path 'urdu-vlm-hallucination' already exists and is not an empty directory.
True


In [ ]:
from huggingface_hub import login
login(token="...")
print("HF login successful")

HF login successful


In [1]:
!pip install -q -U bitsandbytes>=0.46.1

In [2]:
import importlib
import bitsandbytes
importlib.reload(bitsandbytes)
print(bitsandbytes.__version__)  # should show 0.46.1 or higher

0.49.2


In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use only one GPU, avoid split memory

In [2]:
!pip install -q -U bitsandbytes
os.chdir('/kaggle/working')
!git clone https://github.com/ibrahimjohar/urdu-vlm-hallucination.git
os.chdir('urdu-vlm-hallucination')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
Cloning into 'urdu-vlm-hallucination'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 99 (delta 22), reused 43 (delta 13), pack-reused 46 (from 1)
Receiving objects: 100% (99/99), 58.93 MiB | 37.43 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [ ]:
from huggingface_hub import login
login(token="...")

In [4]:
import torch
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
# should show ~14GB free

VRAM free: 15.5 GB


In [ ]:
from huggingface_hub import login
login(token="...")

import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from pathlib import Path
from PIL import Image
import requests

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # reduces fragmentation

class UrduVQADataset(Dataset):
    def __init__(self, json_path, processor, max_length=1024):
        with open(json_path, encoding='utf-8') as f:
            self.data = json.load(f)
        self.processor = processor
        self.max_length = max_length
        self.coco_url = "http://images.cocodataset.org/val2014/{}"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        conv = sample['conversations_ur']
        question = conv[0]['value'].replace('<image>\n', '')
        answer = conv[1]['value']

        image = Image.open(
            requests.get(self.coco_url.format(sample['image']),
                        stream=True, timeout=10).raw
        ).convert('RGB')

        conversation = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }, {
            "role": "assistant",
            "content": [{"type": "text", "text": answer}]
        }]

        prompt = self.processor.apply_chat_template(
            conversation, add_generation_prompt=False
        )
        inputs = self.processor(
            images=image,
            text=prompt,
            return_tensors='pt',
            max_length=self.max_length,
            truncation=False,
            padding='max_length'
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        # FIX 1: LABEL MASKING
        # Previously labels = input_ids for the full sequence, meaning
        # loss was computed on question tokens too. This wastes model
        # capacity and inflates loss. Setting pad tokens to -100 tells
        # PyTorch to ignore them in loss computation — model only learns
        # to predict the answer tokens, which is what we actually want.
        labels = inputs['input_ids'].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        inputs['labels'] = labels

        return inputs

FT_PATH = Path('data/processed/urdu_visual_qa_ft.json')
CHECKPOINT_DIR = Path('outputs/checkpoints/llava_qlora')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8   #effective batch = BATCH_SIZE * GRAD_ACCUM = 8, same as before

# FIX 3: LOWER LEARNING RATE
# 2e-4 was too aggressive — caused the model to overshoot early
# and plateau at 6.6 loss. 1e-4 gives more stable convergence
# especially for cross-lingual transfer tasks.
LR = 1e-4

SAVE_STEPS = 100

#LORA CONFIG
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig, LlavaForConditionalGeneration, AutoProcessor
import torch

# FIX 2: BROADER LORA TARGET MODULES
# Previously only q_proj and v_proj were trained. For cross-lingual
# transfer (English → Urdu), the model needs to update more of its
# attention mechanism. Adding k_proj and o_proj gives LoRA access to
# the full attention block — key for adapting to a new language's
# semantic space without touching the vision encoder.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  #was only q,v
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")

base = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = get_peft_model(base, lora_config)
model.enable_input_require_grads()          # required for grad checkpointing with PEFT
model.gradient_checkpointing_enable()       # trades compute for memory — frees ~30% VRAM
model.print_trainable_parameters()          #sanity check — should show ~1-2% trainable

#load dataset
print("loading dataset...")
dataset = UrduVQADataset(FT_PATH, processor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"samples: {len(dataset)}")
print(f"steps per epoch: {len(loader)}")
print(f"total steps: {len(loader) * EPOCHS}")

#optimizer
optimizer = AdamW(model.parameters(), lr=LR)
model.train()
global_step = 0

print("\nstarting QLoRA training...")
print(f"epochs={EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}\n")

#training loop
for epoch in range(EPOCHS):
    epoch_loss  = 0
    num_batches = 0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        pixel_values = batch['pixel_values'].to('cuda', torch.float16)
        labels = batch['labels'].to('cuda')

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss += outputs.loss.item()
        num_batches += 1
        global_step += 1

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()

        if global_step % 50 == 0:
            avg_loss = epoch_loss / num_batches
            print(f"epoch {epoch+1} | step {global_step} | loss {avg_loss:.4f}")

        if global_step % SAVE_STEPS == 0:
            ckpt_path = CHECKPOINT_DIR / f'checkpoint_step_{global_step}'
            model.save_pretrained(ckpt_path)
            print(f"checkpoint saved: {ckpt_path}")

    avg_epoch_loss = epoch_loss / num_batches
    print(f"\nepoch {epoch+1} complete | avg loss: {avg_epoch_loss:.4f}")

    # push after every epoch — if session dies, epoch N is already safe
    ckpt_path = CHECKPOINT_DIR / f'checkpoint_epoch_{epoch+1}'
    model.save_pretrained(ckpt_path)
    print(f"epoch checkpoint saved: {ckpt_path}")
    model.push_to_hub("ibrahimjohar/llava-1.5-7b-urdu-qlora", commit_message=f"epoch {epoch+1} complete — avg loss {avg_epoch_loss:.4f}")
    print(f"epoch {epoch+1} pushed to HF Hub\n")

#save final
final_path = CHECKPOINT_DIR / 'final'
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"training complete. final model saved to: {final_path}")

model.push_to_hub("ibrahimjohar/llava-1.5-7b-urdu-qlora", commit_message="final model — 2 epochs complete")
processor.push_to_hub("ibrahimjohar/llava-1.5-7b-urdu-qlora")
print("final model pushed to HF Hub")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

epoch 2 pushed to HF Hub

training complete. final model saved to: outputs/checkpoints/llava_qlora/final


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


final model pushed to HF Hub
